# PAPILA Dataset

PAPILA is kept for this project's final test on a separate dataset. This notebook checks its files and labels. It does not train a model, choose model settings, or change the dataset.


## 1. Find the original photographs

Import the tools for reading the ZIP, counting records, and showing one photograph.


In [ ]:
import sys
from collections import Counter
from pathlib import Path
from zipfile import ZipFile
from IPython.display import Image, display
print(f"Notebook Python: {sys.executable}")


Find the local project folder and downloaded ZIP.

In [ ]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

zip_path = project_root / "data/raw/PAPILA.zip"

Stop with a clear message if the ZIP has not been downloaded.

In [ ]:
if not zip_path.is_file():
    raise FileNotFoundError(
        f"Cannot find {zip_path}. Follow the download steps at the top of this notebook."
    )


List the files inside the ZIP. They are all stored under one main folder.


In [ ]:
with ZipFile(zip_path) as archive:
    files = [
        name for name in archive.namelist()
        if not name.endswith("/") and not Path(name).name.startswith(".")
    ]
prefix = files[0].split("/")[0]

Find the original photographs in `FundusImages`. The expert outlines are stored separately.


In [ ]:
image_files = sorted(
    name for name in files
    if name.startswith(f"{prefix}/FundusImages/")
    and Path(name).suffix.lower() == ".jpg"
)
print(f"Original photographs found: {len(image_files)}")

## 2. Read the clinical labels

The two Excel tables contain right-eye (`OD`) and left-eye (`OS`) records. The patient number is in column A and the diagnosis is in column D. The records begin on row 4, after the headings.

The diagnosis codes are 0 = healthy, 1 = glaucoma, and 2 = suspicious. Keep suspicious records separate. We still need to decide how to compare these labels with CHAKSU's labels.


In [ ]:
clinical_files = [
    f"{prefix}/ClinicalData/patient_data_{eye}.xlsx"
    for eye in ["od", "os"]
]
for name in clinical_files:
    print(f"{'FOUND' if name in files else 'MISSING'}: {name}")

Use the same table reader as the full checks, so both read the labels in the same way.


In [ ]:
sys.path.insert(0, str(project_root / "data_pipeline"))
from check_papila import LABEL_NAMES, read_labels

with ZipFile(zip_path) as archive:
    records = read_labels(archive, prefix)

Count patients and eye records. The two eyes of one person share a patient number.


In [ ]:
patient_ids = {record["patient_id"] for record in records}
print(f"Patients in the clinical tables: {len(patient_ids)}")
print(f"Eye records in the clinical tables: {len(records)}")

Count each diagnosis without changing any labels.


In [ ]:
diagnosis_counts = Counter(record["diagnosis"] for record in records)
for value, count in sorted(diagnosis_counts.items()):
    print(f"{value} ({LABEL_NAMES.get(value, 'unknown')}): {count}")

## 3. Match one photograph to its record

Choose the first photograph by filename. Its name contains the patient number and eye. For example, `RET002OD` means the right eye of patient `#002`.


In [ ]:
image_path = image_files[0]
image_id = Path(image_path).stem
matches = [record for record in records if record["image_id"] == image_id]

Make sure this photograph has exactly one matching clinical record.


In [ ]:
if len(matches) != 1:
    raise ValueError(f"Expected one clinical record for {image_id}; found {len(matches)}.")
record = matches[0]

Read the photograph from the ZIP and show it.


In [ ]:
with ZipFile(zip_path) as archive:
    image_bytes = archive.read(image_path)
display(Image(data=image_bytes, width=450))

Show the matching patient, eye, and original diagnosis.

In [ ]:
print(f"Patient: {record['patient_id']}")
print(f"Eye: {record['eye']}")
print(f"Diagnosis: {record['diagnosis']} ({LABEL_NAMES[record['diagnosis']]})")

## 4. Check this photograph's expert outlines

Each photograph should have a cup outline and a disc outline from each of two experts. These are text files listing points along the outlines; they are not mask images.


In [ ]:
contour_files = [
    f"{prefix}/ExpertsSegmentations/Contours/{image_id}_{part}_exp{expert}.txt"
    for part in ["cup", "disc"]
    for expert in [1, 2]
]

Check that all four outline files are present. The full checks also read the points inside them.


In [ ]:
for name in contour_files:
    print(f"{'FOUND' if name in files else 'MISSING'}: {name}")

## 5. Run the full basic checks

Check every photograph, clinical record, and expected outline, then report any problems. When both ZIPs are present, this also looks for identical photo files shared by CHAKSU and PAPILA. It can take a few minutes.

Keep PAPILA for final testing. Decide how to compare the two datasets' labels before looking at the model's test results.


In [ ]:
import runpy

runpy.run_path(str(project_root / "data_pipeline/check_papila.py"), run_name="__main__");

## What was found

- There are 488 photos from 244 patients: 333 healthy, 87 glaucoma, and 68 suspicious.
- All photos opened and matched their labels. All 1,952 required expert outlines were present and passed the basic checks.
- We found 11 extra copies of outline files and left them unchanged.
- We found no identical photo files within PAPILA or shared with the CHAKSU training ZIP.
- Keep PAPILA for final testing. Its clinical labels and CHAKSU's normal/suspect labels mean different things; we still need to decide how to compare them.
